# Socioekonomický index 

Kombinuje sociální a ekonomické faktory a slouží k měření kvality života, nerovností a životní úrovně (počet ekonomicky aktivních obyvatel, RUD, lidé v exekuci). Proměnné byly normalizovány na škálu 0–1 pomocí MinMaxScaler (scikit-learn) a u negativních ukazatelů invertovány tak, aby vyšší hodnota indexu vždy znamenala lepší socioekonomickou situaci. Výsledná škála indexu se pohybuje od 0 do 1.

Vytvoření socioekonimického idexu 
- data z PAQ Research (2024) - příspěvky na bydlení(měsíční průměr), doplatky na bydlení(měsíční průměr), přídavky na děti(měsíční průměr),příspěvky na živobytí(měsíční průměr), nezaměstnanost, hustota zalidnění, počet obyvatel, lidé v exekuci
- Sčítání lidu, bytů a domů 2021 - ekonomicky aktivní obyvatelstvo per capita 
- Rozpočtové určení daní per capita
- přeškálování indexu

In [ ]:
# relevantní socioekonomické údaje PAQ Research 

import pandas as pd

paq = pd.read_csv("dataPAQ_upravena/data_paq_joined.csv")

paq = paq.rename(columns={"Kod_obce": "KodObec"})
paq["KodObec"] = paq['KodObec'].astype(str)
sloupce = ['Prispevky_bydleni_avgMonth_2024',
    'Doplatky_bydleni_avgMonth_2024', 'Pridavky_deti_avgMonth_2024',
    'Prispevky_zivobyti_avgMonth_2024', 'NEZAMESTNANOST',
    'HUSTOTA_ZALIDNENI', 'OBYVATELSTVO_POCET_2024', 'LIDE_EXEKUCE']

paq[sloupce] = paq[sloupce].apply(pd.to_numeric, errors='coerce')

# drop nepotřebné sloupce
paq = paq.drop(columns=['Nazev_obce', 'Kod_okresu', 'Nazev_okresu', 'Kod_kraje',
       'Nazev_kraje',
       'Prispevky_pece_avgMonth_2024', 'Dlouhodoba_nezamestnanost_2024', 'PECOVATELAK',
       'STREDISKO_VOLNY_CAS',
       'Role_obce_na_predskolni_peci_2021', 'VEK_65AVIC', 'VEK_15_64',
       'VEK_0_14', 'DETSKE_SKUPINY', 'DETSKE_HRISTE', 'KULTURAK', 'KINO',
       'KOSTEL', 'OBECNI_POLICIE',
       'POCET_NAROZENYCH_DETI', 'MUZI_SAMOZIVITELE', 'ZENY_SAMOZIVITELKY',
       'DETI_3_4_BEZ_DOCHAZKY_DO_MS'])

# úprava hodnot sloupce 
paq["LIDE_EXEKUCE"] = paq["LIDE_EXEKUCE"].replace("neuvádíme", 0)
paq["LIDE_EXEKUCE"] = pd.to_numeric(paq["LIDE_EXEKUCE"], errors='coerce').fillna(0)


In [ ]:
# ekonomicky aktivní obyvatelstvo SLBD2021 + přepočet na obyvatele

slbd = pd.read_csv("2021_SLDB_OBYVATELSTVO.CSV", encoding="cp1250", usecols=["uzkod", "vse4111", "vse6112","vse6113", "vse6162","vse6163","vse6173"])

slbd["EKONOMICKY_AKTIVNI"] = (
    slbd["vse6112"] +  # aktivní muži
    slbd["vse6113"] +  # aktivní ženy
    slbd["vse6162"] +  # pracující důchodci muži
    slbd["vse6163"] +  # pracující důchodci ženy
    slbd["vse6173"]    # ženy na mateřské
)

slbd = slbd.rename(columns={"vse4111": "celkem_obyv_21", "uzkod":"KodObec"})
slbd["KodObec"] = slbd['KodObec'].astype(str)

# podíl aktivních obyvatel
slbd["EKONOMICKY_AKTIVNI_PODIL"] = slbd["EKONOMICKY_AKTIVNI"] / slbd["celkem_obyv_21"]

In [ ]:
# RUD per capita

RUD = pd.read_csv("RUD_spocitany.csv", usecols=["KodObec", "RUD_celkem"])
RUD["KodObec"] = RUD["KodObec"].astype(str)

chybejici = RUD[~RUD["KodObec"].isin(slbd["KodObec"])]["KodObec"]

# merge se SLBD kvůli počtu obyvatel, který je potřeba pro přepočet na per capita
# v soubrou RUD je komplentní seznam obcí, v slbd je 8 obcí, které nemají údaje ze sčítání (obce v Brdech a Boleticích), proto je merge na RUD, aby zůstal zachovaný správný počet obcí tj 6254)
soceko = RUD.merge(slbd, on="KodObec", how="left")
soceko = soceko.drop(columns=["vse6112","vse6113", "vse6162","vse6163","vse6173"])
soceko[["celkem_obyv_21", "EKONOMICKY_AKTIVNI", "EKONOMICKY_AKTIVNI_PODIL"]] = soceko[["celkem_obyv_21", "EKONOMICKY_AKTIVNI", "EKONOMICKY_AKTIVNI_PODIL"]].fillna(0)
soceko["RUD_percapita"] = soceko["RUD_celkem"] / soceko["celkem_obyv_21"]

soceko["RUD_percapita"] = soceko.apply(
    lambda x: x["RUD_celkem"] / x["celkem_obyv_21"] if x["celkem_obyv_21"] > 0 else None, axis=1)

In [ ]:
# merge paq aby data byla kompletní pro index

soceko = soceko.merge(paq, on="KodObec", how="left")
print(len(soceko))
print(soceko.columns.tolist())

In [ ]:
# škálování a index = čím je vyšší, tím je horší

from sklearn.preprocessing import MinMaxScaler

sloupce_index = ["RUD_percapita", "EKONOMICKY_AKTIVNI_PODIL", "LIDE_EXEKUCE", 
                 "NEZAMESTNANOST", "Prispevky_bydleni_avgMonth_2024", 
                 "Doplatky_bydleni_avgMonth_2024", "Pridavky_deti_avgMonth_2024",
                 "Prispevky_zivobyti_avgMonth_2024"]

scaler = MinMaxScaler()
soceko_scaled = soceko.copy()
soceko_scaled[sloupce_index] = scaler.fit_transform(soceko[sloupce_index])

# invertujeme - vyšší = horší
invertovat = ["LIDE_EXEKUCE", "NEZAMESTNANOST", "Prispevky_bydleni_avgMonth_2024", 
              "Doplatky_bydleni_avgMonth_2024", "Pridavky_deti_avgMonth_2024",
              "Prispevky_zivobyti_avgMonth_2024"]

soceko_scaled[invertovat] = 1 - soceko_scaled[invertovat]

# index jako průměr
soceko_scaled["index_socioekonomicky"] = soceko_scaled[sloupce_index].mean(axis=1)

#soceko_scaled[["KodObec", "index_socioekonomicky"]].describe()

soceko_scaled = soceko_scaled.drop(columns=['RUD_celkem', 'celkem_obyv_21', 'EKONOMICKY_AKTIVNI',
       'EKONOMICKY_AKTIVNI_PODIL', 'RUD_percapita',
       'Prispevky_bydleni_avgMonth_2024', 'Doplatky_bydleni_avgMonth_2024',
       'Pridavky_deti_avgMonth_2024', 'Prispevky_zivobyti_avgMonth_2024',
       'NEZAMESTNANOST', 'HUSTOTA_ZALIDNENI', 'OBYVATELSTVO_POCET_2024',
       'LIDE_EXEKUCE'])

soceko_scaled["KodObec"] = soceko_scaled["KodObec"].astype(str) 
soceko_scaled.to_csv("index_socioekonomicky.csv", index=False, sep=",")